In [1]:
from google.colab import drive
drive.mount('/content/drive')

exec(open('/content/drive/MyDrive/brain_tumor_classification/setup_colab.py').read())

Mounted at /content/drive

  Step 1 — Detecting environment
  Running in Colab : True
  Python version   : 3.12.13
  OS               : Linux 6.6.113+

  Step 2 — Setting up project path
  Project root : /content/drive/MyDrive/brain_tumor_classification

  Step 3 — Creating project structure
  Folder structure created.

  /content/drive/MyDrive/brain_tumor_classification/
    ├── data/
    ├── src/
    ├── experiments/
    ├── outputs/
    ├── notebooks/
    ├── logs/

  Step 4 — Installing dependencies
  Installing from: /content/drive/MyDrive/brain_tumor_classification/requirements.txt
  This may take 2–3 minutes...

  $ pip install -q -r /content/drive/MyDrive/brain_tumor_classification/requirements.txt
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 86.8 MB/s eta 0:00:00

  All dependencies installed.

  Step 5 — Dataset
  Dataset already present on Drive — skipping download.

  Step 6 — Dataset verification
  Training/glioma           1400 images   OK
  Training/meningioma    

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

import torch, timm, albumentations, cv2, sklearn, matplotlib, numpy
print("=" * 45)
print(f"  PyTorch        : {torch.__version__}")
print(f"  CUDA           : {torch.cuda.is_available()}")
print(f"  GPU            : {torch.cuda.get_device_name(0)}")
print(f"  NumPy          : {numpy.__version__}")
print(f"  OpenCV         : {cv2.__version__}")
print(f"  timm           : {timm.__version__}")
print(f"  albumentations : {albumentations.__version__}")
print(f"  scikit-learn   : {sklearn.__version__}")
print("=" * 45)
print("Phase 1 complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  PyTorch        : 2.10.0+cu128
  CUDA           : True
  GPU            : NVIDIA H100 80GB HBM3
  NumPy          : 2.0.2
  OpenCV         : 4.13.0
  timm           : 1.0.25
  albumentations : 2.0.8
  scikit-learn   : 1.6.1
Phase 1 complete!


In [3]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

from src.dataset import create_dataloaders, get_dataset_info

DATA_DIR = '/content/drive/MyDrive/brain_tumor_classification/data'

train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir=DATA_DIR,
    image_size=224,
    batch_size=32,
)

get_dataset_info(info)

images, labels = next(iter(train_loader))
print(f"Batch shape : {images.shape}")
print(f"Labels      : {labels[:8].tolist()}")

  Dataset Summary
  Classes      : glioma, meningioma, notumor, pituitary
  Image size   : 224x224
  Batch size   : 32

  Train set    :  4480 images  (140 batches)
  Val set      :  1120 images  (35 batches)
  Test set     :  1600 images  (50 batches)
  Total        :  7200 images

  Class → Index mapping:
    0 → glioma
    1 → meningioma
    2 → notumor
    3 → pituitary
Batch shape : torch.Size([32, 3, 224, 224])
Labels      : [2, 1, 0, 0, 0, 3, 3, 1]


In [4]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

from src.models import build_model
from src.train import get_device, train_model

print("ResNet files imported successfully!")

ResNet files imported successfully!


In [5]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

from config import get_default_config
from src.models import build_model
from src.train import get_device

cfg = get_default_config()
cfg.name = 'baseline_resnet50'
cfg.model.backbone = 'resnet50'
cfg.model.pretrained = True
cfg.model.num_classes = 4
cfg.data.image_size = 224

device = get_device()
model = build_model(cfg).to(device)

print("Device:", device)
print("Backbone:", cfg.model.backbone)
print("Num classes:", cfg.model.num_classes)
print(model.__class__.__name__)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Device: cuda
Backbone: resnet50
Num classes: 4
BrainTumorClassifier


In [6]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

from config import get_default_config
from src.dataset import create_dataloaders

cfg = get_default_config()
cfg.name = 'baseline_resnet50'
cfg.model.backbone = 'resnet50'
cfg.model.pretrained = True
cfg.model.num_classes = 4
cfg.data.image_size = 224

DATA_DIR = '/content/drive/MyDrive/brain_tumor_classification/data'

train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir=DATA_DIR,
    image_size=cfg.data.image_size,
    batch_size=cfg.data.batch_size,
)

print("Dataloaders ready!")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Dataloaders ready!
Train batches: 140
Val batches: 35
Test batches: 50


In [7]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

from config import get_default_config
from src.dataset import create_dataloaders
from src.train import train_model, get_device

cfg = get_default_config()
cfg.name = 'baseline_resnet50'
cfg.model.backbone = 'resnet50'
cfg.model.pretrained = True
cfg.model.num_classes = 4
cfg.data.image_size = 224

DATA_DIR = '/content/drive/MyDrive/brain_tumor_classification/data'

train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir=DATA_DIR,
    image_size=cfg.data.image_size,
    batch_size=cfg.data.batch_size,
)

device = get_device()
model, history, best_ckpt = train_model(cfg, train_loader, val_loader, device=device)

print("Training finished!")
print("Best checkpoint:", best_ckpt)

Config saved to: /content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50/config.yaml


Starting training
Experiment     : baseline_resnet50
Backbone       : resnet50
Device         : cuda
Monitor        : val_f1
Experiment dir : /content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50

[Phase change] epoch=1  phase=warmup  lr=0.001
Trainable params: 8,196 / 23,516,228


/content/drive/MyDrive/brain_tumor_classification/src/train.py:230: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=bool(cfg.train.use_amp and device.type == 'cuda'))


  0%|          | 0/140 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

...

  0%|          | 0/35 [00:00<?, ?it/s]

Epoch 17/30 | train_loss=0.9294 | val_loss=0.7449 | val_acc=82.59% | val_f1=82.28%
  -> No improvement. Early-stop counter: 6/7


  0%|          | 0/140 [00:00<?, ?it/s]

  0%|          | 0/35 [00:00<?, ?it/s]

Epoch 18/30 | train_loss=0.9277 | val_loss=0.7389 | val_acc=82.32% | val_f1=81.94%
  -> No improvement. Early-stop counter: 7/7

Early stopping triggered.

Training complete.
Best checkpoint : /content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50/best_model.pth
Metrics CSV      : /content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50/metrics.csv
Training finished!
Best checkpoint: /content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50/best_model.pth


In [8]:
import os

exp_dir = '/content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50'
print(os.listdir(exp_dir))

['config.yaml', 'best_model.pth', 'metrics.csv']


In [9]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/brain_tumor_classification')

from config import get_default_config
from src.dataset import create_dataloaders
from src.models import build_model
from src.train import load_checkpoint, evaluate_model, get_device

cfg = get_default_config()
cfg.name = 'baseline_resnet50'
cfg.model.backbone = 'resnet50'
cfg.model.pretrained = True
cfg.model.num_classes = 4
cfg.data.image_size = 224

DATA_DIR = '/content/drive/MyDrive/brain_tumor_classification/data'

train_loader, val_loader, test_loader, info = create_dataloaders(
    data_dir=DATA_DIR,
    image_size=cfg.data.image_size,
    batch_size=cfg.data.batch_size,
)

best_ckpt = '/content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50/best_model.pth'

device = get_device()
model = build_model(cfg)
model = load_checkpoint(model, best_ckpt, device=device)

test_metrics = evaluate_model(model, test_loader, device=device)
print("Test metrics:", test_metrics)

Eval:   0%|          | 0/50 [00:00<?, ?it/s]

Test metrics: {'loss': 0.71222, 'acc': 76.3125, 'f1': 75.1508}


In [10]:
import pandas as pd

metrics_path = '/content/drive/MyDrive/brain_tumor_classification/experiments/baseline_resnet50/metrics.csv'
df = pd.read_csv(metrics_path)
print(df)

    epoch             phase        lr  train_loss  train_acc  train_f1  \
0       1            warmup  0.000905    1.202558    55.5580   55.0742   
1       2            warmup  0.000655    1.039549    63.9509   63.6409   
2       3            warmup  0.000346    0.981525    67.1652   66.8494   
3       4            warmup  0.000096    0.948075    68.7723   68.5957   
4       5            warmup  0.000001    0.947750    69.0402   68.8697   
5       6  partial_finetune  0.000099    0.945543    69.0848   68.8474   
6       7  partial_finetune  0.000095    0.940939    68.5938   68.4490   
7       8  partial_finetune  0.000089    0.941932    69.6652   69.4944   
8       9  partial_finetune  0.000081    0.930568    69.9107   69.6096   
9      10  partial_finetune  0.000072    0.930441    70.2902   70.1209   
10     11  partial_finetune  0.000062    0.925569    70.0670   69.9124   
11     12  partial_finetune  0.000051    0.933184    69.5312   69.3020   
12     13  partial_finetune  0.000039 